# EC7 · Laboratorio 09: partes 2 y 3

Este notebook desarrolla **preprocesamiento y Bag of Words** y **búsqueda booleana lineal e índice invertido** con la base PostgreSQL local `ec7`.

Se asume que `noticias(id, contenido, bag_of_words)` y `stopwords(word)` ya existen y tienen los datos de la parte 1. No se crea la base ni se cargan nuevamente las noticias. La única escritura es la actualización de `noticias.bag_of_words` solicitada en la parte 2.

Las funciones de coseno que aparecen adelantadas en la plantilla de la parte 3 corresponden a la parte 4. Se conservan `idf` y `length` como pide la construcción del índice en la parte 3, sin implementar las búsquedas de las partes posteriores.

## Conexión a la base existente

Instala las dependencias en el entorno de tu kernel con `%pip install -r requirements.txt` si aún no están disponibles. Completa el archivo local `.env` a partir de `.env.example` y ejecuta las celdas en orden. La contraseña se lee localmente; `.env` está excluido del repositorio.

Se utilizan los parámetros `PGHOST`, `PGPORT`, `PGDATABASE`, `PGUSER` y `PGPASSWORD`. También se admite la autenticación configurada en `.pgpass`. No hace falta descargar recursos adicionales de NLTK: el tokenizador por expresión regular y Snowball funcionan localmente.

In [1]:
import json
import math
import os
import re
import unicodedata
from collections import Counter
from contextlib import closing

import pandas as pd
import psycopg2
from psycopg2.extras import Json, execute_values
from dotenv import load_dotenv
from IPython.display import display
from nltk.stem.snowball import SnowballStemmer
from nltk.tokenize import RegexpTokenizer

load_dotenv(override=False)


def connect_db():
    """Conectarse a ec7 sin guardar credenciales en el notebook."""
    config = {
        'dbname': os.getenv('PGDATABASE', 'ec7'),
        'user': os.getenv('PGUSER', 'postgres'),
        'host': os.getenv('PGHOST', 'localhost'),
        'port': os.getenv('PGPORT', '5432'),
        'connect_timeout': 5,
    }
    if os.getenv('PGPASSWORD'):
        config['password'] = os.environ['PGPASSWORD']
    return psycopg2.connect(**config)


def read_dataframe(sql):
    """Leer con cursor para cerrar siempre la conexión y evitar avisos de pandas."""
    with closing(connect_db()) as conn:
        with conn.cursor() as cur:
            cur.execute(sql)
            columns = [description[0] for description in cur.description]
            rows = cur.fetchall()
    return pd.DataFrame(rows, columns=columns)


def fetch_data():
    return read_dataframe('SELECT id, contenido FROM noticias ORDER BY id;')


def fetch_stopwords():
    dataframe = read_dataframe('SELECT word FROM stopwords ORDER BY word;')
    return {
        unicodedata.normalize('NFC', word).strip().lower()
        for word in dataframe['word']
        if isinstance(word, str) and word.strip()
    }

In [2]:
noticias_df = fetch_data()
STOPWORDS = fetch_stopwords()
if noticias_df.empty:
    raise ValueError('La tabla noticias está vacía. Completa la carga de la parte 1 antes de continuar.')
if not STOPWORDS:
    raise ValueError('La tabla stopwords está vacía. Completa la carga de la parte 1 antes de continuar.')
print(f'Base: {os.getenv("PGDATABASE", "ec7")}')
print(f'Noticias: {len(noticias_df):,} | Stopwords: {len(STOPWORDS):,}')
display(noticias_df.head(3))

Base: ec7
Noticias: 1,217 | Stopwords: 608


,id,contenido
0,1,Durante el foro La banca articulador empresari...
1,2,El regulador de valores de China dijo el domin...
2,3,En una industria históricamente masculina como...


## Parte 2 · Preprocesamiento y Bag of Words

El flujo es: normalización Unicode NFC y minúsculas → tokenización de palabras → eliminación de stopwords → stemming en español → conteo de frecuencias. Se mantienen tildes y `ñ` al tokenizar; Snowball obtiene las raíces. Se descartan números y signos de puntuación. Los textos nulos o vacíos producen `{}`.

Las stopwords se eliminan **antes** del stemming para compararlas con las palabras originales normalizadas. Un BoW es un diccionario `{raíz: frecuencia}` y no conserva el orden de las palabras.

In [3]:
tokenizer = RegexpTokenizer(r'[^\W\d_]+')
stemmer = SnowballStemmer('spanish')


def preprocess(text):
    """Minúsculas, tokenización, eliminación de stopwords y stemming."""
    if not isinstance(text, str):
        return []
    normalized = unicodedata.normalize('NFC', text).lower()
    return [
        stemmer.stem(token)
        for token in tokenizer.tokenize(normalized)
        if token not in STOPWORDS
    ]


def compute_bow(text):
    """Devolver las frecuencias de las raíces del documento."""
    return dict(Counter(preprocess(text)))

In [4]:
ejemplo = 'La tecnología y las tecnologías impulsan la economía. Tecnología, 2026.'
print('Texto:', ejemplo)
print('Tokens procesados:', preprocess(ejemplo))
print('Bag of Words:', compute_bow(ejemplo))
assert compute_bow(None) == {}
assert compute_bow('') == {}
assert compute_bow('!!! 123') == {}
assert compute_bow('Tecnología tecnología') == {stemmer.stem('tecnología'): 2}

Texto: La tecnología y las tecnologías impulsan la economía. Tecnología, 2026.
Tokens procesados: ['tecnolog', 'tecnolog', 'impuls', 'econom', 'tecnolog']
Bag of Words: {'tecnolog': 3, 'impuls': 1, 'econom': 1}


### Guardar y recuperar el BoW

`update_bow_in_db` actualiza únicamente `bag_of_words`, conserva los identificadores y usa una transacción: si ocurre un error o falta algún identificador, se deshace toda la actualización. Repetir la celda recalcula los mismos BoW sin insertar documentos.

El PDF usa el nombre `noticias_bow` en un comentario, pero define la columna `bag_of_words` en `noticias`. Por eso se consulta **`noticias`**. `fetch_data_bow()` recupera todos los documentos para buscar en toda la colección; se muestran solamente las primeras 10 filas.

In [5]:
def update_bow_in_db(dataframe):
    required = {'id', 'contenido'}
    if not required.issubset(dataframe.columns):
        raise ValueError('Se requieren las columnas id y contenido.')
    if dataframe['id'].isna().any() or dataframe['id'].duplicated().any():
        raise ValueError('Los identificadores deben ser únicos y no nulos.')
    updates = [
        (int(doc_id), Json(compute_bow(content)))
        for doc_id, content in dataframe[['id', 'contenido']].itertuples(index=False, name=None)
    ]
    if not updates:
        return 0
    sql = """
        UPDATE noticias AS n
        SET bag_of_words = v.bow::jsonb
        FROM (VALUES %s) AS v(id, bow)
        WHERE n.id = v.id
        RETURNING n.id;
    """
    with closing(connect_db()) as conn:
        with conn:
            with conn.cursor() as cur:
                updated = execute_values(cur, sql, updates, page_size=250, fetch=True)
                if len(updated) != len(updates):
                    raise ValueError('No se encontraron todos los IDs; se revierte la actualización.')
    return len(updated)


def fetch_data_bow():
    return read_dataframe(
        'SELECT id, contenido, bag_of_words FROM noticias ORDER BY id;'
    )

In [6]:
updated_count = update_bow_in_db(noticias_df)
noticias_bow_df = fetch_data_bow()
assert len(noticias_bow_df) == len(noticias_df)
assert noticias_bow_df['bag_of_words'].map(lambda bow: isinstance(bow, dict)).all()
print(f'BoW actualizados: {updated_count:,}')
display(noticias_bow_df.head(10))

BoW actualizados: 1,217


,id,contenido,bag_of_words
0,1,Durante el foro La banca articulador empresari...,"{'aca': 1, 'adn': 1, 'cas': 1, 'for': 1, 'ret'..."
1,2,El regulador de valores de China dijo el domin...,"{'dat': 1, 'deb': 1, 'inc': 1, 'med': 1, 'mes'..."
2,3,En una industria históricamente masculina como...,"{'go': 1, 'us': 3, 'air': 1, 'baj': 1, 'ceo': ..."
3,4,Con el dato de marzo el IPC interanual encaden...,"{'agu': 1, 'car': 2, 'cas': 1, 'dat': 2, 'ine'..."
4,5,Ayer en Cartagena se dio inicio a la versión n...,"{'agu': 1, 'cup': 2, 'deb': 2, 'etc': 1, 'for'..."
5,6,El emprendimiento del sector inmobiliario Rent...,"{'us': 2, 'big': 1, 'ceo': 1, 'dat': 1, 'sum':..."
6,7,El país tiene pendiente generar más y mejores ...,"{'ve': 1, 'baj': 1, 'cas': 2, 'dat': 1, 'dej':..."
7,8,En el 'podcast' tenemos el placer de entrevist...,"{'agil': 4, 'habl': 1, 'mund': 1, 'nach': 3, '..."
8,9,"El factor humano, normalmente identificado com...","{'u': 1, 'ali': 2, 'bas': 1, 'cas': 2, 'cin': ..."
9,10,Los perfiles que se demandan en un banco han e...,"{'are': 1, 'cab': 1, 'cas': 1, 'dat': 2, 'pud'..."


## Parte 3 · Búsqueda booleana lineal e índice invertido

- `AND`: documentos que contienen ambos términos.
- `OR`: documentos que contienen al menos uno.
- `AND-NOT`: documentos del primer conjunto que no están en el segundo.

Las consultas permiten paréntesis. `AND` y `AND-NOT` tienen igual prioridad y se evalúan de izquierda a derecha; ambos tienen prioridad sobre `OR`. Los operadores aceptan mayúsculas o minúsculas. Cada término pasa por el mismo preprocesamiento de las noticias. Se rechazan consultas vacías, sintaxis inválida y términos eliminados como stopwords; no se admite `NOT` aislado.

Un único analizador construye un árbol de la consulta. Las dos búsquedas evalúan ese mismo árbol con estrategias diferentes, lo que permite comparar sus resultados directamente.

In [7]:
QUERY_TOKEN = re.compile(r'AND-NOT\b|AND\b|OR\b|[^\W\d_]+|[()]', re.IGNORECASE)
OPERATORS = {'AND', 'OR', 'AND-NOT'}


def normalize_query_term(word):
    terms = preprocess(word)
    if len(terms) != 1:
        raise ValueError(f'El término {word!r} debe producir una raíz y no ser una stopword.')
    return terms[0]


def parse_boolean_query(query):
    """Analizador descendente: OR < AND/AND-NOT; los paréntesis agrupan."""
    if not isinstance(query, str) or not query.strip():
        raise ValueError('La consulta no puede estar vacía.')
    query = unicodedata.normalize('NFC', query)
    tokens = []
    previous_end = 0
    for match in QUERY_TOKEN.finditer(query):
        if query[previous_end:match.start()].strip():
            raise ValueError('La consulta contiene caracteres no admitidos.')
        value = match.group()
        tokens.append(value.upper() if value.upper() in OPERATORS else value)
        previous_end = match.end()
    if query[previous_end:].strip():
        raise ValueError('La consulta contiene caracteres no admitidos.')
    position = 0

    def atom():
        nonlocal position
        if position >= len(tokens):
            raise ValueError('Falta un término en la consulta.')
        token = tokens[position]
        position += 1
        if token == '(':
            node = expression()
            if position >= len(tokens) or tokens[position] != ')':
                raise ValueError('Falta cerrar un paréntesis.')
            position += 1
            return node
        if token == ')' or token in OPERATORS:
            raise ValueError(f'Se esperaba un término, se encontró {token!r}.')
        return ('TERM', normalize_query_term(token))

    def conjunction():
        nonlocal position
        node = atom()
        while position < len(tokens) and tokens[position] in {'AND', 'AND-NOT'}:
            operator = tokens[position]
            position += 1
            node = (operator, node, atom())
        return node

    def expression():
        nonlocal position
        node = conjunction()
        while position < len(tokens) and tokens[position] == 'OR':
            position += 1
            node = ('OR', node, conjunction())
        return node

    tree = expression()
    if position != len(tokens):
        raise ValueError('Hay términos sin operador o paréntesis sobrantes.')
    return tree

### 3.1 · Búsqueda lineal

Se recorre cada documento y se verifica la presencia de las raíces en su BoW. Se devuelven los IDs ordenados, sin duplicados. La frecuencia no afecta una búsqueda booleana: basta con que sea mayor que cero.

In [8]:
def boolean_search_linear(query, dataframe):
    tree = parse_boolean_query(query)

    def satisfies(node, bow):
        operator = node[0]
        if operator == 'TERM':
            return bow.get(node[1], 0) > 0
        left = satisfies(node[1], bow)
        if operator == 'AND':
            return left and satisfies(node[2], bow)
        if operator == 'OR':
            return left or satisfies(node[2], bow)
        return left and not satisfies(node[2], bow)

    results = []
    for doc_id, bow in dataframe[['id', 'bag_of_words']].itertuples(index=False, name=None):
        if not isinstance(bow, dict):
            raise ValueError(f'El documento {doc_id} no tiene un BoW válido; ejecuta la parte 2.')
        if satisfies(tree, bow):
            results.append(int(doc_id))
    return sorted(set(results))

In [9]:
consultas_lineales = [
    'economía AND inflación',
    'banca OR finanzas',
    'tecnología AND-NOT inflación',
]
for consulta in consultas_lineales:
    resultados = boolean_search_linear(consulta, noticias_bow_df)
    print(f'{consulta}: {len(resultados)} documentos; primeros IDs: {resultados[:10]}')

economía AND inflación: 244 documentos; primeros IDs: [5, 7, 18, 21, 24, 32, 33, 34, 37, 39]
banca OR finanzas: 547 documentos; primeros IDs: [1, 2, 5, 9, 10, 12, 13, 15, 16, 17]
tecnología AND-NOT inflación: 306 documentos; primeros IDs: [12, 13, 14, 15, 16, 17, 19, 20, 23, 28]


### 3.2 · Índice invertido

`index[raíz]` contiene una lista de tuplas `(doc_id, tf)` ordenada por ID; `tf` es la frecuencia absoluta del término. `_docs` guarda los textos para mostrar fragmentos.

También se calculan las estructuras pedidas en la plantilla: `idf(t) = ln(N / df(t))` y `length[d] = sqrt(sum((tf(t,d) * idf(t)) ** 2))`. `N` es el total de documentos, incluidos los de BoW vacío. Un documento sin términos tiene norma cero. Estos valores se almacenan, aunque la búsqueda booleana usa solamente presencia de términos.

`L("tecnología")` normaliza y obtiene la raíz antes de consultar el índice; así, las palabras de las pruebas del PDF coinciden con las claves generadas por stemming.

In [10]:
class InvertedIndex:
    def __init__(self):
        self.index = {}
        self.idf = {}
        self.length = {}
        self._docs = {}

    def build_from_db(self):
        dataframe = fetch_data_bow()
        new_index = {}
        new_docs = {}
        for doc_id, content, bow in dataframe.itertuples(index=False, name=None):
            doc_id = int(doc_id)
            if not isinstance(bow, dict):
                raise ValueError(f'El documento {doc_id} no tiene BoW; ejecuta la parte 2.')
            new_docs[doc_id] = content if isinstance(content, str) else ''
            for term, tf in bow.items():
                if not isinstance(tf, int) or isinstance(tf, bool) or tf <= 0:
                    raise ValueError(f'Frecuencia inválida en el documento {doc_id}: {term!r}.')
                new_index.setdefault(term, []).append((doc_id, tf))
        for postings in new_index.values():
            postings.sort(key=lambda posting: posting[0])
        total_docs = len(new_docs)
        new_idf = {
            term: math.log(total_docs / len(postings))
            for term, postings in new_index.items()
        }
        squared_lengths = dict.fromkeys(new_docs, 0.0)
        for term, postings in new_index.items():
            for doc_id, tf in postings:
                squared_lengths[doc_id] += (tf * new_idf[term]) ** 2
        self.index = new_index
        self.idf = new_idf
        self.length = {doc_id: math.sqrt(value) for doc_id, value in squared_lengths.items()}
        self._docs = new_docs
        return self

    def L(self, word):
        return list(self.index.get(normalize_query_term(word), []))

    def showDocument(self, doc_id, max_chars=80):
        if max_chars < 0:
            raise ValueError('max_chars debe ser no negativo.')
        text = self._docs.get(doc_id, '')
        return text[:max_chars] + ('...' if len(text) > max_chars else '')

    def showDocuments(self, posting_list, max_chars=80):
        return [self.showDocument(doc_id, max_chars) for doc_id, *_ in posting_list]

### Operaciones sobre las listas de postings

Se usan dos punteros para recorrer listas ya ordenadas, sin volver a recorrer la colección. Cada operación cuesta `O(m + n)`, donde `m` y `n` son los tamaños de las listas de entrada.

En un resultado compuesto, el segundo campo suma las frecuencias de los términos que contribuyeron a ese documento. Es un dato auxiliar: las operaciones y la comparación con la búsqueda lineal dependen únicamente del ID. `AND_NOT` conserva la frecuencia de la lista izquierda.

In [11]:
def AND(list1, list2):
    result = []
    i = j = 0
    while i < len(list1) and j < len(list2):
        id1, tf1 = list1[i]
        id2, tf2 = list2[j]
        if id1 == id2:
            result.append((id1, tf1 + tf2))
            i += 1
            j += 1
        elif id1 < id2:
            i += 1
        else:
            j += 1
    return result


def OR(list1, list2):
    result = []
    i = j = 0
    while i < len(list1) and j < len(list2):
        id1, tf1 = list1[i]
        id2, tf2 = list2[j]
        if id1 == id2:
            result.append((id1, tf1 + tf2))
            i += 1
            j += 1
        elif id1 < id2:
            result.append(list1[i])
            i += 1
        else:
            result.append(list2[j])
            j += 1
    result.extend(list1[i:])
    result.extend(list2[j:])
    return result


def AND_NOT(list1, list2):
    result = []
    i = j = 0
    while i < len(list1) and j < len(list2):
        id1 = list1[i][0]
        id2 = list2[j][0]
        if id1 == id2:
            i += 1
            j += 1
        elif id1 < id2:
            result.append(list1[i])
            i += 1
        else:
            j += 1
    result.extend(list1[i:])
    return result


def boolean_search_index(query, inverted_index):
    tree = parse_boolean_query(query)
    operations = {'AND': AND, 'OR': OR, 'AND-NOT': AND_NOT}

    def evaluate(node):
        if node[0] == 'TERM':
            # El parser ya hizo stemming; no se procesa la raíz por segunda vez.
            return inverted_index.index.get(node[1], [])
        return operations[node[0]](evaluate(node[1]), evaluate(node[2]))

    return [doc_id for doc_id, _ in evaluate(tree)]

In [12]:
idx = InvertedIndex().build_from_db()
print(f'Documentos indexados: {len(idx._docs):,}')
print(f'Términos distintos: {len(idx.index):,}')
print('Primeros postings de tecnología:', idx.L('tecnología')[:10])

Documentos indexados: 1,217
Términos distintos: 13,339
Primeros postings de tecnología: [(12, 4), (13, 1), (14, 1), (15, 2), (16, 1), (17, 1), (19, 7), (20, 1), (23, 5), (28, 4)]


### Pruebas del PDF y tres pruebas adicionales

Las tres primeras expresiones son las de la guía; se agregan otras tres combinando los operadores. Para cada expresión se verifica que la composición explícita de postings, la búsqueda con índice y la búsqueda lineal devuelvan exactamente los mismos IDs. Se muestran el número de coincidencias y hasta dos fragmentos, para mantener legible la salida.

In [13]:
pruebas = [
    ('sostenibilidad AND (ambiente AND renovables)',
     lambda: AND(idx.L('sostenibilidad'), AND(idx.L('ambiente'), idx.L('renovables')))),
    ('tecnología AND (banca OR finanzas)',
     lambda: AND(idx.L('tecnología'), OR(idx.L('banca'), idx.L('finanzas')))),
    ('economía AND-NOT inflación',
     lambda: AND_NOT(idx.L('economía'), idx.L('inflación'))),
    ('banca AND finanzas',
     lambda: AND(idx.L('banca'), idx.L('finanzas'))),
    ('(economía OR tecnología) AND-NOT inflación',
     lambda: AND_NOT(OR(idx.L('economía'), idx.L('tecnología')), idx.L('inflación'))),
    ('(sostenibilidad OR renovables) AND ambiente',
     lambda: AND(OR(idx.L('sostenibilidad'), idx.L('renovables')), idx.L('ambiente'))),
]
resumen = []
for consulta, ejecutar_postings in pruebas:
    postings = ejecutar_postings()
    ids_explicitos = [doc_id for doc_id, _ in postings]
    lineal = boolean_search_linear(consulta, noticias_bow_df)
    invertido = boolean_search_index(consulta, idx)
    assert lineal == invertido == ids_explicitos, f'Resultados distintos: {consulta}'
    resumen.append({'consulta': consulta, 'documentos': len(lineal), 'resultados_iguales': True})
    print(f'\n{consulta}: {len(lineal)} documentos')
    for fragmento in idx.showDocuments(postings[:2], max_chars=120):
        print(' •', fragmento)

display(pd.DataFrame(resumen))
print('Las seis pruebas devuelven los mismos IDs con ambos métodos.')


sostenibilidad AND (ambiente AND renovables): 9 documentos
 • La importancia que cada día más empresas le otorgan al tema de la sostenibilidad como parte esencial de su modelo de neg...
 • Esta tipología de energía, clave para la sostenibilidad del planeta, nace de la fusión nuclear de helio e hidrógeno. Por...

tecnología AND (banca OR finanzas): 174 documentos
 • BBVA ha presentado BBVA Spark, su propuesta integral de servicios financieros para acompañar a las compañías innovadoras...
 • Con acciones que se comprometen con el desarrollo sostenible, ambas organizaciones trabajan  en conjunto para buscar sol...

economía AND-NOT inflación: 358 documentos
 • Las plataformas de domicilios fueron uno de los jugadores más importantes de la pandemia pero acabando el confinamiento ...
 • El regulador bancario de China dijo que las recientes medidas para frenar el monopolio de las empresas de tecnología fin...

banca AND finanzas: 76 documentos
 • Con acciones que se comprometen con el desar

,consulta,documentos,resultados_iguales
0,sostenibilidad AND (ambiente AND renovables),9,True
1,tecnología AND (banca OR finanzas),174,True
2,economía AND-NOT inflación,358,True
3,banca AND finanzas,76,True
4,(economía OR tecnología) AND-NOT inflación,541,True
5,(sostenibilidad OR renovables) AND ambiente,41,True


Las seis pruebas devuelven los mismos IDs con ambos métodos.


### Verificación de casos límite

Se comprueban listas vacías, ausencia de términos en el vocabulario, mayúsculas, paréntesis, prioridad de operadores, BoW vacíos, conteos repetidos y rechazo de consultas inválidas. Estas comprobaciones no escriben en la base de datos.

In [14]:
a = [(1, 2), (3, 1)]
b = [(2, 4), (3, 5)]
assert AND(a, b) == [(3, 6)]
assert OR(a, b) == [(1, 2), (2, 4), (3, 6)]
assert AND_NOT(a, b) == [(1, 2)]
assert AND(a, []) == []
assert OR([], b) == b
assert AND_NOT(a, []) == a
assert AND_NOT([], b) == []
assert idx.L('zzzxqvpalabrainexistente') == []

mini_df = pd.DataFrame([
    {'id': 1, 'bag_of_words': compute_bow('economía')},
    {'id': 2, 'bag_of_words': compute_bow('tecnología inflación')},
    {'id': 3, 'bag_of_words': compute_bow('tecnología banca')},
    {'id': 4, 'bag_of_words': {}},
])
mini_idx = InvertedIndex()
for row in mini_df.itertuples(index=False):
    for term, tf in row.bag_of_words.items():
        mini_idx.index.setdefault(term, []).append((row.id, tf))
for consulta, esperado in [
    ('economía OR tecnología AND inflación', [1, 2]),
    ('(economía OR tecnología) AND inflación', [2]),
    ('tecnología AND-NOT inflación', [3]),
    ('ECONOMÍA or TECNOLOGÍA', [1, 2, 3]),
    ('zzzxqvpalabrainexistente', []),
    ('(economía OR tecnología) AND-NOT (inflación OR banca)', [1]),
]:
    assert boolean_search_linear(consulta, mini_df) == esperado
    assert boolean_search_index(consulta, mini_idx) == esperado

for consulta_invalida in ['', 'AND economía', 'economía OR', '(economía',
                          'economía)', 'economía tecnología', 'economía $ inflación',
                          'economía AND ()']:
    try:
        parse_boolean_query(consulta_invalida)
    except ValueError:
        continue
    raise AssertionError(f'Se aceptó una consulta inválida: {consulta_invalida!r}')

print('Casos límite y sintaxis verificados correctamente.')

Casos límite y sintaxis verificados correctamente.


## Resultado de las partes 2 y 3

El BoW queda persistido en `noticias.bag_of_words`, y el índice invertido se construye en memoria a partir de esos valores. Las pruebas comparan IDs, ya que la búsqueda booleana no asigna puntuaciones de similitud. Si cambian las noticias o las stopwords, vuelve a ejecutar desde la carga de datos para recalcular los BoW y reconstruir el índice.

Referencias de implementación: [Snowball en NLTK](https://www.nltk.org/api/nltk.stem.snowball.html) y [actualizaciones por lotes con psycopg2](https://www.psycopg.org/docs/extras.html#psycopg2.extras.execute_values).